# Chinese to English Word-by-Word Translation

For the BabyLM 2026 multilingual track. Team: Ajay, Tyler, Libby.

---

### What we're doing

- Translating Chinese text into English, but word-by-word so it keeps the Chinese word order.
- So you get "broken" English like "I at school study" instead of "I study at school."
- We train models on this later so they act like a Chinese speaker using English.

### The data

- We use `BabyLM-community/babylm-zho` (the official Chinese data from BabyBabelLM).
- It's gated, so you need to request access on HuggingFace and make a token first.
- About 204k documents: child speech, kids' books, school/educational text, subtitles, etc.

### How we cleaned it (Sections 2 and 3)

- Took out the speaker tags like `*ADULT:` and `*TARGET_CHILD:` but kept what they said.
- Threw away math junk, empty lines, and lines with no Chinese.
- Dropped the subtitles (they're news broadcasts, too formal, and gave messy translations).
- Split the long paragraphs into single sentences so the model gets clean short lines.
- Left us with about 1.3 million clean Chinese lines.

### How we translate (Section 4)

- Using gpt-4.1-mini. We switched to it because it follows the line-by-line rules better than 4o-mini.
- The prompt just tells it: translate word-by-word, keep the meaning, one line in = one line out, don't skip or reorder.
- It saves to Google Drive as it goes and picks up where it left off if Colab disconnects.
- We split the file into pieces so all three of us can run different parts at the same time.

### What the output looks like

- "Usually have pay attention NBA?"
- "One by one sheep rush out gate, jumping run toward boundless grassland."
- That's the Chinese-style English we want.

### What's next (not in this notebook yet)

- Put everyone's pieces back into one file.
- Final cleanup (Section 5).
- Train the models and run the BabyLM tests.

---

**How to run:** go top to bottom. Sections 1 to 3 only need to be run once (the cleaned files are already on Drive), so if you're just translating, start at Section 4 and set your own line range.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Download dataset + setup


In [ ]:
!pip install huggingface_hub datasets -q

from huggingface_hub import login
login(token="hf____------")  # paste your hugging face token here

from datasets import load_dataset
zh = load_dataset("BabyLM-community/babylm-zho")
print(zh)

print("\nCategory distribution:")
from collections import Counter
cats = Counter(zh['train']['category'])
for cat, count in cats.most_common():
    print(f"  {cat}: {count:,} docs")

print("\nSample from category child-directed-speech:")
for ex in zh['train']:
    if ex['category'] == 'child-directed-speech':
        print(ex['text'][:300])
        break

print("\nSample from category child-books:")
for ex in zh['train']:
    if ex['category'] == 'child-books':
        print(ex['text'][:300])
        break

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


data/train-00000-of-00002.parquet:   0%|          | 0.00/315M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/34.5M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'doc-id', 'category', 'data-source', 'script', 'age-estimate', 'license', 'misc', 'num-tokens', 'language'],
        num_rows: 203891
    })
})

Category distribution:
  educational: 74,930 docs
  child-directed-speech: 42,884 docs
  subtitles: 40,586 docs
  child-books: 25,175 docs
  child-available-speech: 20,249 docs
  child-wiki: 67 docs

Sample from category child-directed-speech:
*ADULT:	这 个 是 什么 呀.
*TARGET_CHILD:	苹果.
*ADULT:	对 这 是 苹果.
*ADULT:	那 你 能不能 向 小 羊 一样 介绍 苹果 呢.
*ADULT:	小 羊 不 认识 苹果.
*ADULT:	你 能不能 跟 它 说 说.
*ADULT:	什么 是 苹果 呀.
*ADULT:	来 试 一下 吧.
*ADULT:	我 们 一起 好不好.
*ADULT:	先 说 说 苹果 是 什么 形状 的.
*TARGET_CHILD:	圆形.
*ADULT:	对 苹果 是 圆形 的.
*ADULT:	它 是 什么 颜色 的 啊.
*TARGET_CHILD:	红色

Sample from category child-books:
一只只羊儿涌出了圈门,蹦跳着奔向无边的草原。牧民们跨上骏马,追赶那欢乐的羊群。远处,一群群羊儿像朵朵白云在飘动,蓝天下回荡着牧羊人的歌声。从文中找出下列词语的反义词。进--出有--无近--远黑--白


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted!


In [ ]:
import os
DRIVE = "/content/drive/MyDrive/BabyLM_Chinese_English_Translation"

# Save with category info so we can filter/analyze later
print("Saving correct Chinese dataset to Drive...")

with open(f"{DRIVE}/chinese_train_correct.txt", "w", encoding="utf-8") as f:
    for example in zh['train']:
        f.write(example['text'].strip() + "\n")

import os
size = os.path.getsize(f"{DRIVE}/chinese_train_correct.txt") / (1024*1024)
print(f"Saved! Size: {size:.1f} MB")

# Count lines
with open(f"{DRIVE}/chinese_train_correct.txt") as f:
    lines = sum(1 for _ in f)
print(f"Total lines: {lines:,}")

# Show samples from each category
print("\n=== SAMPLES BY CATEGORY ===")
for cat in ['educational', 'child-books', 'child-directed-speech', 'subtitles']:
    for ex in zh['train']:
        if ex['category'] == cat:
            print(f"\n[{cat}]")
            print(ex['text'][:200])
            break

Saving correct Chinese dataset to Drive...
Saved! Size: 495.0 MB
Total lines: 2,077,484

=== SAMPLES BY CATEGORY ===

[educational]
==自然数==
===导言===
你会数数吗?也许你会不屑地说:"我早就会了!"那么你最大能数到多少呢?你会做加法、减法、乘法和除法吗?你知道分配律是什么吗?如果还不是特别清楚的话,就来读一读这一章的内容吧。

让我们还是回到数数的问题上来。什么是数数呢?我们先看一个例子:小明有一篮苹果,小红也有一篮苹果,可是谁的更多一些呢?我们就需要比一比。一个办法是小明和小红每次各吃一个苹果,看谁的先吃完,谁

[child-books]
一只只羊儿涌出了圈门,蹦跳着奔向无边的草原。牧民们跨上骏马,追赶那欢乐的羊群。远处,一群群羊儿像朵朵白云在飘动,蓝天下回荡着牧羊人的歌声。从文中找出下列词语的反义词。进--出有--无近--远黑--白

[child-directed-speech]
*ADULT:	这 个 是 什么 呀.
*TARGET_CHILD:	苹果.
*ADULT:	对 这 是 苹果.
*ADULT:	那 你 能不能 向 小 羊 一样 介绍 苹果 呢.
*ADULT:	小 羊 不 认识 苹果.
*ADULT:	你 能不能 跟 它 说 说.
*ADULT:	什么 是 苹果 呀.
*ADULT:	来 试 一下 吧.
*ADULT:	我 们 一起 好不好.
*ADULT:	

[subtitles]
来自南方都市报播报武汉大学老牌坊被撞损一事持续引发关注 六月八十日 南都记者从武汉市公安局洪山分局一位工作人员处获悉 撞损牌坊的肇事司机因涉嫌过失损毁文物罪被依法刑事立案 南都此前报道 六月七日 武汉大学官方微博发布说明 位于洪山区街道口区域劝业场的武汉大学老牌坊被附近工地施工的混凝土搅拌车碰擦 导致部分受损 该老牌坊建于一九三四年 是全国重点保护文物 武汉大学早期建筑之一 文保部门已紧急为受损老


## 2. Clean the source (remove junk, exclude subtitles)


In [ ]:
import re, os

DRIVE = "/content/drive/MyDrive/BabyLM_Chinese_English_Translation"
output_file = f"{DRIVE}/chinese_train_sentence_level_v2.txt"

def clean_line(line):
    line = line.strip()
    # Remove CHILDES speaker markers but keep the text
    line = re.sub(r'^\*(ADULT|TARGET_CHILD|MOTHER|FATHER|INVESTIGATOR|TEACHER|CHILD|EXPERIMENTER|CHI|MOT|FAT)\s*:\s*', '', line)
    line = line.replace('\t', ' ').strip()
    if len(line) < 4:
        return None
    # Must have Chinese characters
    if not any('\u4e00' <= c <= '\u9fff' for c in line):
        return None
    # No LaTeX/math
    if any(t in line for t in ['$$', '\\frac', '\\sqrt', '\\left', '${']):
        return None
    # Must be mostly letters
    letters = sum(c.isalpha() for c in line)
    if letters < len(line) * 0.3:
        return None
    return line

# EXCLUDE subtitles — only keep good categories
GOOD_CATEGORIES = ['child-books', 'child-directed-speech', 'educational',
                   'child-available-speech', 'child-wiki']

print("Building clean source file (no subtitles)...")
stats = {}
total_kept = 0

with open(output_file, "w", encoding="utf-8") as f_out:
    for category in GOOD_CATEGORIES:
        cat_kept = 0
        for ex in zh['train']:
            if ex['category'] != category:
                continue
            for line in ex['text'].split('\n'):
                cleaned = clean_line(line)
                if cleaned:
                    f_out.write(cleaned + "\n")
                    cat_kept += 1
        stats[category] = cat_kept
        total_kept += cat_kept
        print(f"  {category}: {cat_kept:,} lines")

size = os.path.getsize(output_file) / (1024*1024)
print(f"\nTotal clean lines (no subtitles): {total_kept:,}")
print(f"File size: {size:.1f} MB")

# Show one sample from each category
print("\n=== SAMPLES ===")
for category in GOOD_CATEGORIES:
    for ex in zh['train']:
        if ex['category'] != category:
            continue
        lines = ex['text'].split('\n')
        for line in lines:
            cleaned = clean_line(line)
            if cleaned and len(cleaned) > 10:
                print(f"\n[{category}]")
                print(f" {cleaned[:120]}")
                break
        break

Building clean source file (no subtitles)...
  child-books: 25,291 lines
  child-directed-speech: 480,646 lines
  educational: 251,452 lines
  child-available-speech: 542,714 lines
  child-wiki: 436 lines

Total clean lines (no subtitles): 1,300,539
File size: 133.2 MB

=== SAMPLES ===

[child-books]
 一只只羊儿涌出了圈门,蹦跳着奔向无边的草原。牧民们跨上骏马,追赶那欢乐的羊群。远处,一群群羊儿像朵朵白云在飘动,蓝天下回荡着牧羊人的歌声。从文中找出下列词语的反义词。进--出有--无近--远黑--白

[child-directed-speech]
 这 个 是 什么 呀.

[educational]
 你会数数吗?也许你会不屑地说:"我早就会了!"那么你最大能数到多少呢?你会做加法、减法、乘法和除法吗?你知道分配律是什么吗?如果还不是特别清楚的话,就来读一读这一章的内容吧。

[child-available-speech]
 爱数智慧语音采集二零一九年十月三十一日。

[child-wiki]
 狮,俗称狮子,是非洲的标志性野生动物。因为它们的力量与健美的外观,狮子常被非洲部落崇敬与赞美。狮子也是唯一的大规模群居的大型猫科动物。另外,它们的吼声可是猫科动物中最响亮的,最远八公里外都能听到!


## 3. Split into sentences


In [ ]:
import re
import os
from tqdm import tqdm

DRIVE = "/content/drive/MyDrive/BabyLM_Chinese_English_Translation"

input_file = f"{DRIVE}/chinese_train_sentence_level_v2.txt"
output_file = f"{DRIVE}/chinese_train_sentence_level.txt"

with open(input_file, "r", encoding="utf-8") as f:
    lines = [l.strip() for l in f if l.strip()]

new_lines = []

for line in tqdm(lines):

    # split on Chinese sentence endings
    pieces = re.split(r'[。！？!?；;]', line)

    for p in pieces:
        p = p.strip()

        # keep only useful Chinese sentences
        if len(p) < 4:
            continue

        if not any('\u4e00' <= c <= '\u9fff' for c in p):
            continue

        # remove weird website markers
        if "童话网" in p:
            continue

        # remove chapter markers
        if p.startswith("#"):
            continue

        new_lines.append(p)

with open(output_file, "w", encoding="utf-8") as f:
    for line in new_lines:
        f.write(line + "\n")

print("\nDone.")
print(f"Original lines : {len(lines):,}")
print(f"Sentence lines : {len(new_lines):,}")

lengths = [len(x) for x in new_lines]

print(f"Average length : {sum(lengths)/len(lengths):.1f}")
print(f"Max length     : {max(lengths):,}")

print("\nLongest 10 lengths:")
for x in sorted(lengths, reverse=True)[:10]:
    print(x)

100%|██████████| 1300539/1300539 [00:05<00:00, 258696.78it/s]



Done.
Original lines : 1,300,539
Sentence lines : 2,294,544
Average length : 21.3
Max length     : 2,954

Longest 10 lengths:
2954
2375
1303
1089
1081
1010
1009
1009
1009
989


##########Optional preprocessing cell — already completed###############

In [ ]:
import re
from tqdm import tqdm

DRIVE = "/content/drive/MyDrive/BabyLM_Chinese_English_Translation"

input_file = f"{DRIVE}/chinese_train_sentence_level.txt"
output_file = f"{DRIVE}/chinese_train_sentence_level_v2.txt"

with open(input_file, "r", encoding="utf-8") as f:
    lines = [x.strip() for x in f if x.strip()]

cleaned = []

for line in tqdm(lines):

    # Throw away extremely long lines
    if len(line) > 500:
        continue

    # Remove weird separators
    line = line.replace("#", " ")
    line = line.replace("//", " ")

    line = re.sub(r"\s+", " ", line).strip()

    if len(line) < 4:
        continue

    cleaned.append(line)

with open(output_file, "w", encoding="utf-8") as f:
    for line in cleaned:
        f.write(line + "\n")

lengths = [len(x) for x in cleaned]

print("\nDone")
print(f"Total lines: {len(cleaned):,}")
print(f"Average length: {sum(lengths)/len(lengths):.1f}")
print(f"Max length: {max(lengths)}")

print("\nTop 10 longest:")
for x in sorted(lengths, reverse=True)[:10]:
    print(x)

100%|██████████| 2294544/2294544 [00:04<00:00, 471866.04it/s]



Done
Total lines: 2,294,519
Average length: 21.3
Max length: 497

Top 10 longest:
497
490
490
482
480
476
476
474
474
472


## 4. Translate

Change `my_lines = all_lines[:400000]` to your assigned slice (e.g. `[400000:800000]`). Rename the output/progress files with your name. Saves to Drive and resumes automatically if disconnected.


In [ ]:
!pip install --upgrade openai -q
import openai
client = openai.AsyncOpenAI(api_key="sk-proj-...")   # <-- your OpenAI key


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 27.1 MB/s eta 0:00:00


In [ ]:
import os
import asyncio
from tqdm import tqdm

DRIVE = "/content/drive/MyDrive/BabyLM_Chinese_English_Translation"

with open(f"{DRIVE}/chinese_train_sentence_level_v2.txt", "r", encoding="utf-8") as f:
    all_lines = [l.strip() for l in f if l.strip()]

print(f"Total clean sentence lines: {len(all_lines):,}")

my_lines = all_lines[400000]
print(f"Lines selected: {len(my_lines):,}")

output_file = f"{DRIVE}/translated_part_400k_gpt41mini_ORDERED_RESTART.txt"
progress_file = f"{DRIVE}/progress_400k_gpt41mini_ORDERED_RESTART.txt"

open(output_file, "a", encoding="utf-8").close()
print("Output file created:", os.path.exists(output_file))

start_idx = 0
if os.path.exists(progress_file):
    with open(progress_file, "r") as f:
        start_idx = int(f.read().strip())
    print(f"Resuming from line {start_idx:,}")
else:
    print("Starting fresh")

PROMPT = """
Translate each numbered Mandarin Chinese sentence into English word-by-word.

Preserve the Mandarin word order and sentence structure as much as possible, even if the English sounds broken or unnatural.

Keep location phrases, time phrases, adverbs, and modifiers in the same relative position as in the Mandarin sentence.

Do not rewrite into fluent English.

If a Mandarin multi-word expression is best translated as one English word, use the single English word.

Return exactly one numbered English output line for each numbered input line.

Do not skip, merge, or reorder lines.
"""

async def translate_batch(lines):
    numbered = "\n".join([f"{i+1}. {line}" for i, line in enumerate(lines)])

    for attempt in range(5):
        try:
            response = await client.chat.completions.create(
                model="gpt-4.1-mini",
                messages=[
                    {"role": "system", "content": PROMPT},
                    {"role": "user", "content": numbered}
                ],
                max_completion_tokens=2048,
                temperature=0
            )

            result = response.choices[0].message.content.strip()

            if len(result) < 5:
                print("Warning: suspiciously short response")

            translated = []

            for line in result.split("\n"):
                line = line.strip()
                if not line:
                    continue

                if line and line[0].isdigit() and ". " in line:
                    line = line.split(". ", 1)[1]

                translated.append(line)

            while len(translated) < len(lines):
                translated.append("")

            blank_count = translated.count("")
            if blank_count > 0:
                print(f"Warning: {blank_count} blank translations out of {len(lines)}")

            return translated[:len(lines)]

        except openai.RateLimitError:
            wait_time = (attempt + 1) * 10
            print(f"Rate limit encountered. Waiting {wait_time} seconds...")
            await asyncio.sleep(wait_time)

        except Exception as e:
            print(f"Error: {e}")
            await asyncio.sleep(5)

    return [""] * len(lines)

async def main():
    batch_size = 10
    max_concurrent = 2
    chunk_size = 50

    semaphore = asyncio.Semaphore(max_concurrent)

    async def process_batch(lines):
        async with semaphore:
            result = await translate_batch(lines)
            await asyncio.sleep(0.1)
            return result

    async def indexed_process_batch(index, batch):
        result = await process_batch(batch)
        return index, result

    print(f"Lines remaining: {len(my_lines) - start_idx:,}")

    for chunk_start in range(start_idx, len(my_lines), chunk_size * batch_size):
        chunk_end = min(chunk_start + chunk_size * batch_size, len(my_lines))
        chunk_lines = my_lines[chunk_start:chunk_end]

        batches = [
            chunk_lines[i:i + batch_size]
            for i in range(0, len(chunk_lines), batch_size)
        ]

        tasks = [
            indexed_process_batch(i, batch)
            for i, batch in enumerate(batches)
        ]

        results = [None] * len(tasks)

        for coro in tqdm(
            asyncio.as_completed(tasks),
            total=len(tasks),
            desc=f"Lines {chunk_start:,}-{chunk_end:,}"
        ):
            idx, result = await coro
            results[idx] = result

        with open(output_file, "a", encoding="utf-8") as out_f:
            for batch_result in results:
                for line in batch_result:
                    out_f.write(line.strip() + "\n")

            out_f.flush()
            os.fsync(out_f.fileno())

        with open(progress_file, "w") as pf:
            pf.write(str(chunk_end))

        with open(output_file, "r", encoding="utf-8") as check_f:
            saved_lines = sum(1 for _ in check_f)

        print(f"Saved to Drive: {chunk_end:,} / {len(my_lines):,}")
        print(f"Output lines now: {saved_lines:,}")

        if saved_lines != chunk_end:
            raise RuntimeError(
                f"STOPPING: output lines ({saved_lines}) != progress ({chunk_end})"
            )

    print("\nTranslation complete!")

await main()

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/BabyLM_Chinese_English_Translation/chinese_train_sentence_level_v2.txt'

## 5. Check samples + final clean


In [ ]:
import random

DRIVE = "/content/drive/MyDrive/BabyLM_Chinese_English_Translation"

source_file = f"{DRIVE}/chinese_train_sentence_level_v2.txt"
translated_file = f"{DRIVE}/translated_part_400k_gpt41mini_ORDERED_RESTART.txt"

with open(source_file, "r", encoding="utf-8") as f:
    src = [x.strip() for x in f if x.strip()]

with open(translated_file, "r", encoding="utf-8") as f:
    trg = [x.rstrip("\n") for x in f]

print("Translated lines:", len(trg))

for idx in random.sample(range(len(trg)), 20):
    print("\n" + "="*80)
    print("INDEX:", idx)
    print("SRC :", src[idx])
    print("TRG :", trg[idx])

Translated lines: 312500

INDEX: 73878
SRC : 第二天,宋献策乔装扮成一测字老先生,混入北京城内,在皇宫附近,摆下测字摊,一幅白布招牌迎风摇摆,上书:"鬼谷②为师,管格③是友
TRG : The next day, Song Xiance disguised as a fortune-telling old master, mixed into Beijing city, near the palace, set up fortune-telling stall, a white cloth signboard waving in the wind, written: "Guigu is teacher, Guange is friend."

INDEX: 220850
SRC : "小螃蟹急忙跑去告诉小章鱼说:"不好了,不好了,小鱼受伤了
TRG : Little crab hurriedly ran to tell little octopus saying: "Not good, not good, little fish injured.

INDEX: 202369
SRC : " 小布猴和小布狗溜到地上,拉着手,蹑手蹑脚的来到厨房里,借着窗外的月光,看见一个一身灰色的毛,尖嘴巴嘴巴上有几根胡子,圆圆的两只耳朵,一条细细的尾巴的小动物,正在从一个碗柜里拖出一块鱼
TRG : " Little cloth monkey and little cloth dog sneaked to ground, holding hands, tiptoed to kitchen, borrowing moonlight outside window, saw a little animal with gray fur all over, pointed snout with several whiskers on snout, two round ears, a thin tail, was dragging a piece of fish out from a cupboard

INDEX: 108695
SRC : 小狐嚼着胡萝卜,泪珠悄悄滚下脸颊
TRG : Little fox chews carrot, tears 

In [ ]:
import re

DRIVE = "/content/drive/MyDrive/BabyLM_Chinese_English_Translation"

input_file = f"{DRIVE}/translated_part_400k_gpt41mini_ORDERED_RESTART.txt"
output_file = f"{DRIVE}/english_babylm.txt"

with open(input_file, "r", encoding="utf-8") as f:
    lines = f.readlines()

clean = []

for line in lines:
    line = line.strip()

    if not line:
        continue

    line = re.sub(r"\s+", " ", line)

    clean.append(line)

with open(output_file, "w", encoding="utf-8") as f:
    for line in clean:
        f.write(line + "\n")

print("Saved:", output_file)
print("Total sentences:", len(clean))

Saved: /content/drive/MyDrive/BabyLM_Chinese_English_Translation/english_babylm.txt
Total sentences: 312478


In [ ]:
DRIVE = "/content/drive/MyDrive/BabyLM_Chinese_English_Translation"

with open(f"{DRIVE}/english_babylm.txt",
          "r",
          encoding="utf-8") as f:

    lines = [x.strip() for x in f]

tokens = sum(len(x.split()) for x in lines)

print("Sentences:", len(lines))
print("Total words:", tokens)
print("Average words/sentence:", tokens / len(lines))

Sentences: 312478
Total words: 4978972
Average words/sentence: 15.933832141782782


for subtitle data translation (use this dataset only beacsue i shuffled it  and converted into chanks of 60 chars in one line and keep the batch size = 5 or smaller ) and i also changed the parser in the translation of this .

In [ ]:
import os
import asyncio
from tqdm import tqdm
import re

DRIVE = "/content/drive/MyDrive/BabyLM_Chinese_English_Translation"

with open(f"{DRIVE}/chinese_train_subtitles_chunks_60chars_period_shuffled_seed42.txt", "r", encoding="utf-8") as f: # use this dataset only for subtitles file
    all_lines = [l.strip() for l in f if l.strip()]

print(f"Total subtitle chunks: {len(all_lines):,}")

my_lines = all_lines[0:300000]
print(f"Lines selected: {len(my_lines):,}")

output_file = f"{DRIVE}/translated_subtitles_0_300000_gpt41mini_60chars_shuffled_seed42.txt"# make this files accoring your part ajay[0:600k]
progress_file = f"{DRIVE}/progress_subtitles_0_300000_gpt41mini_60chars_shuffled_seed42.txt"# make thsi files accoring your part ajay[0:600k]

open(output_file, "a", encoding="utf-8").close()
print("Output file created:", os.path.exists(output_file))

start_idx = 0
if os.path.exists(progress_file):
    with open(progress_file, "r") as f:
        start_idx = int(f.read().strip())
    print(f"Resuming from line {start_idx:,}")
else:
    print("Starting fresh")

PROMPT = """
Translate each numbered Mandarin Chinese sentence into English word-by-word.

Preserve the Mandarin word order and sentence structure as much as possible, even if the English sounds broken or unnatural.

Keep location phrases, time phrases, adverbs, and modifiers in the same relative position as in the Mandarin sentence.

Do not rewrite into fluent English.

If a Mandarin multi-word expression is best translated as one English word, use the single English word.

Return exactly one numbered English output line for each numbered input line.

Do not skip, merge, or reorder lines.
"""

async def translate_batch(lines):
    numbered = "\n".join([f"{i+1}. {line}" for i, line in enumerate(lines)])

    for attempt in range(5):
        try:
            response = await client.chat.completions.create(
                model="gpt-4.1-mini",
                messages=[
                    {"role": "system", "content": PROMPT},
                    {"role": "user", "content": numbered}
                ],
                max_completion_tokens=2048,
                temperature=0
            )

            result = response.choices[0].message.content.strip()

            if len(result) < 5:
                print("Warning: suspiciously short response")

            translated_by_id = {}

            for line in result.split("\n"):
                line = line.strip()
                if not line:
                    continue

                match = re.match(r"^(\d+)\.\s*(.*)$", line)

                if match:

                    num = match.group(1)

                    text = match.group(2)

                    if num.isdigit():
                        idx = int(num)

                        if 1 <= idx <= len(lines):
                            text = text.strip()

                            # If the model echoes Chinese first and then English with the same number,
                            # this keeps the later line for that number.
                            translated_by_id[idx] = text

            translated = [
                translated_by_id[i]
                for i in range(1, len(lines) + 1)
                if i in translated_by_id
            ]

            if len(translated) != len(lines):
                print("\n" + "=" * 100)
                print("FAILED BATCH DEBUG")
                print("Expected:", len(lines))
                print("Got:", len(translated))

                print("\nINPUT LINES:")
                for i, src in enumerate(lines, 1):
                    print(f"{i}. {src}")

                print("\nRAW MODEL RESPONSE:")
                print(result)
                print("=" * 100 + "\n")

                print(f"Retrying: expected {len(lines)}, got {len(translated)}")
                await asyncio.sleep(3)
                continue

            return translated

        except openai.RateLimitError:
            wait_time = (attempt + 1) * 10
            print(f"Rate limit encountered. Waiting {wait_time} seconds...")
            await asyncio.sleep(wait_time)

        except Exception as e:
            print(f"Error: {e}")
            await asyncio.sleep(5)

    raise RuntimeError("Failed to translate batch after 5 attempts.")

async def main():
    batch_size = 5
    max_concurrent = 2
    chunk_size = 50

    semaphore = asyncio.Semaphore(max_concurrent)

    async def process_batch(lines):
        async with semaphore:
            result = await translate_batch(lines)
            await asyncio.sleep(0.1)
            return result

    async def indexed_process_batch(index, batch):
        result = await process_batch(batch)
        return index, result

    print(f"Lines remaining: {len(my_lines) - start_idx:,}")

    for chunk_start in range(start_idx, len(my_lines), chunk_size * batch_size):
        chunk_end = min(chunk_start + chunk_size * batch_size, len(my_lines))
        chunk_lines = my_lines[chunk_start:chunk_end]

        batches = [
            chunk_lines[i:i + batch_size]
            for i in range(0, len(chunk_lines), batch_size)
        ]

        tasks = [
            indexed_process_batch(i, batch)
            for i, batch in enumerate(batches)
        ]

        results = [None] * len(tasks)

        for coro in tqdm(
            asyncio.as_completed(tasks),
            total=len(tasks),
            desc=f"Lines {chunk_start:,}-{chunk_end:,}"
        ):
            idx, result = await coro
            results[idx] = result

        with open(output_file, "a", encoding="utf-8") as out_f:
            for batch_result in results:
                for line in batch_result:
                    out_f.write(line.strip() + "\n")

            out_f.flush()
            os.fsync(out_f.fileno())

        with open(progress_file, "w") as pf:
            pf.write(str(chunk_end))

        with open(output_file, "r", encoding="utf-8") as check_f:
            saved_lines = sum(1 for _ in check_f)

        print(f"Saved to Drive: {chunk_end:,} / {len(my_lines):,}")
        print(f"Output lines now: {saved_lines:,}")

        if saved_lines != chunk_end:
            raise RuntimeError(
                f"STOPPING: output lines ({saved_lines}) != progress ({chunk_end})"
            )

    print("\nTranslation complete!")

await main()